# RobotMath2030 — Tiny World Model + Planning (Ch.10)

Latent dynamics, imagination MPC, open-loop drift.

Full chapter: [10_tiny_world_model](../chapters/10_tiny_world_model/)

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    !git clone -q https://github.com/rsasaki0109/RobotMath2030.git 2>/dev/null || true
    %cd RobotMath2030
except ImportError:
    root = Path.cwd()
    if not (root / 'robotmath').exists() and (root.parent / 'robotmath').exists():
        %cd ..

!pip install -q -e ".[torch]"
%matplotlib inline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from miniworlds.grid_world import WALL, GOAL, GridWorld
from robotmath.world_models import WorldModelConfig, rollout_env, train_world_model

env = GridWorld(layout='easy')
data = env.collect_random_dataset(n_transitions=1200, seed=0)
model, losses = train_world_model(data, WorldModelConfig(epochs=50, hidden=64, seed=0))

env.reset()
env.agent = (1, 1)
goal_obs = env.observation()
closed_path, closed_ok = rollout_env(env, model, goal_obs, replan=True, horizon=6, n_candidates=48, seed=1)
open_path, open_ok = rollout_env(env, model, goal_obs, replan=False, horizon=5, n_candidates=64, seed=1)

print(f'Closed-loop replan success: {closed_ok}')
print(f'Open-loop imagination success: {open_ok}')

def draw(ax, path, title):
    grid = np.zeros_like(env.grid, dtype=float)
    grid[env.grid == WALL] = 1.0
    grid[env.grid == GOAL] = 0.6
    ax.imshow(grid, origin='upper', cmap='Greys', vmin=0, vmax=1)
    ys = [p[0] for p in path]
    xs = [p[1] for p in path]
    ax.plot(xs, ys, '-o', color='C0', markersize=4)
    ax.scatter([env.goal[1]], [env.goal[0]], c='gold', s=80, edgecolors='k', zorder=5)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
draw(axes[0], closed_path, 'Closed-loop replan')
draw(axes[1], open_path, 'Open-loop drift')
axes[2].semilogy(losses, 'C0-o', markersize=3)
axes[2].set_title('Training loss')
axes[2].set_xlabel('epoch')
axes[2].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()